In [1]:
"""
CAIM
=====

# CAIM (class-attribute interdependence maximization) algorithm for
        supervised discretization

.. note::
    "L. A. Kurgan and K. J. Cios (2004), CAIM discretization algorithm in
    IEEE Transactions on Knowledge and Data Engineering, vol. 16, no. 2, pp. 145-153, Feb. 2004.
    doi: 10.1109/TKDE.2004.1269594"
    .. _a link: http://ieeexplore.ieee.org/document/1269594/

.. module:: caimcaim
   :platform: Unix, Windows
   :synopsis: A simple, but effective discretization algorithm

"""


import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class CAIMD(BaseEstimator, TransformerMixin):

    def __init__(self, categorical_features='auto'):

        if isinstance(categorical_features, str):
            self._features = categorical_features
            self.categorical = None
        elif (isinstance(categorical_features, list)) or (isinstance(categorical_features, np.ndarray)):
            self._features = None
            self.categorical = categorical_features
        else:
            raise CategoricalParamException(
                "Wrong type for 'categorical_features'. Expected 'auto', an array of indicies or labels.")

    def fit(self, X, y):

        self.split_scheme = dict()
        if isinstance(X, pd.DataFrame):
            # self.indx = X.index
            # self.columns = X.columns
            if isinstance(self._features, list):
                self.categorical = [X.columns.get_loc(label) for label in self._features]
            X = X.values
            y = y.values
        if self._features == 'auto':
            self.categorical = self.check_categorical(X, y)
        categorical = self.categorical
        print('Categorical', categorical)

        min_splits = np.unique(y).shape[0]

        for j in range(X.shape[1]):
            if j in categorical:
                continue
            xj = X[:, j]
            xj = xj[np.invert(np.isnan(xj))]
            new_index = xj.argsort()
            xj = xj[new_index]
            yj = y[new_index]
            allsplits = np.unique(xj)[1:-1].tolist()
            global_caim = -1
            mainscheme = [xj[0], xj[-1]]
            best_caim = 0
            k = 1
            while (k <= min_splits) or ((global_caim < best_caim) and (allsplits)):
                split_points = np.random.permutation(allsplits).tolist()
                best_scheme = None
                best_point = None
                best_caim = 0
                k = k + 1
                while split_points:
                    scheme = mainscheme[:]
                    sp = split_points.pop()
                    scheme.append(sp)
                    scheme.sort()
                    c = self.get_caim(scheme, xj, yj)
                    if c > best_caim:
                        best_caim = c
                        best_scheme = scheme
                        best_point = sp
                if (k <= min_splits) or (best_caim > global_caim):
                    mainscheme = best_scheme
                    global_caim = best_caim
                    try:
                        allsplits.remove(best_point)
                    except ValueError:
                        raise NotEnoughPoints('The feature #' + str(j) + ' does not have' +
                                              ' enough unique values for discretization!' +
                                              ' Add it to categorical list!')

            self.split_scheme[j] = mainscheme
            print('#', j, ' GLOBAL CAIM ', global_caim)
        return self

    def transform(self, X):

        if isinstance(X, pd.DataFrame):
            self.indx = X.index
            self.columns = X.columns
            X = X.values
        X_di = X.copy()
        categorical = self.categorical

        scheme = self.split_scheme
        for j in range(X.shape[1]):
            if j in categorical:
                continue
            sh = scheme[j]
            sh[-1] = sh[-1] + 1
            xj = X[:, j]
            # xi = xi[np.invert(np.isnan(xi))]
            for i in range(len(sh) - 1):
                ind = np.where((xj >= sh[i]) & (xj < sh[i + 1]))[0]
                X_di[ind, j] = i
        if hasattr(self, 'indx'):
            return pd.DataFrame(X_di, index=self.indx, columns=self.columns)
        return X_di

    def fit_transform(self, X, y):

        self.fit(X, y)
        return self.transform(X)

    def get_caim(self, scheme, xi, y):
        sp = self.index_from_scheme(scheme[1:-1], xi)
        sp.insert(0, 0)
        sp.append(xi.shape[0])
        n = len(sp) - 1
        isum = 0
        for j in range(n):
            init = sp[j]
            fin = sp[j + 1]
            Mr = xi[init:fin].shape[0]
            val, counts = np.unique(y[init:fin], return_counts=True)
            maxr = counts.max()
            isum = isum + (maxr / Mr) * maxr
        return isum / n

    def index_from_scheme(self, scheme, x_sorted):
        split_points = []
        for p in scheme:
            split_points.append(np.where(x_sorted > p)[0][0])
        return split_points

    def check_categorical(self, X, y):
        categorical = []
        ny2 = 2 * np.unique(y).shape[0]
        for j in range(X.shape[1]):
            xj = X[:, j]
            xj = xj[np.invert(np.isnan(xj))]
            if np.unique(xj).shape[0] < ny2:
                categorical.append(j)
        return categorical


class CategoricalParamException(Exception):
    pass


class NotEnoughPoints(Exception):
    pass

In [2]:
"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\gamma.csv")
X = data[['fLength', 'fWidth', "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\winequality-red.csv")
X = data[['fixed acidity', 'volatile acidity', "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\RiceCameo.csv")
X = data[['Area Integer', 'Permimeter Real', "Major Axis", "Minor Axis", "EccentricityReal", "Convex Area", "Extent Real"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\winequality-white.csv")
X = data[['fixed acidity', 'volatile acidity', "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\glass_identification.csv")
X = data[["refractive index","Sodium","Magnesium","Aluminum","Silicon","Potassium","Calcium","Barium","Iron"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\iris.csv")
X = data[["sepal_length","sepal_width","petal_length","petal_width"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\letter_recognition.csv")
X = data[["x-box","y-box","width","high","onpix","x-bar","y-bar","x2bar","y2bar","xybar","x2ybr","xy2br","x-ege","xegvy","y-ege","yegvx"]]
y = data['class']
"""

"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\seeds.csv")
X = data[["area","perimeter","compactness","lengthKernel","widthKernel","asymmetryCoefficient","lengthGroove"]]
y = data['class']
"""


data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\Dry_Bean_dataset.csv")
X = data[["Area","Perimeter","MajorAxisLength","MinorAxisLength","AspectRation","Eccentricity","ConvexArea","EquivDiameter","Extent","Solidity","roundness","Compactness",
          "ShapeFactor1","ShapeFactor2","ShapeFactor3","ShapeFactor4"]]
y = data['class']


"""
data = pd.read_csv("C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\bases de datos\\yeast.csv")
X = data[["mcg","gvh","alm","mit","erl","pox","vac","nuc"]]
y = data['class']
"""


import time


inicio = time.time()
caim = CAIMD()
x_disc = caim.fit_transform(X, y)
fin = time.time()

print(fin - inicio)


Categorical []
# 0  GLOBAL CAIM  812.6976142563037
# 1  GLOBAL CAIM  843.8515690108683
# 2  GLOBAL CAIM  776.5638509186058
# 3  GLOBAL CAIM  720.7728284420648
# 4  GLOBAL CAIM  797.4551358403436
# 5  GLOBAL CAIM  797.4551358403436
# 6  GLOBAL CAIM  817.4878085796882
# 7  GLOBAL CAIM  812.6976142563037
# 8  GLOBAL CAIM  279.52494728685036
# 9  GLOBAL CAIM  284.048778786912
# 10  GLOBAL CAIM  719.755216837073
# 11  GLOBAL CAIM  808.1031114670612
# 12  GLOBAL CAIM  721.7496270961627
# 13  GLOBAL CAIM  729.5569869315999
# 14  GLOBAL CAIM  808.1031114670612
# 15  GLOBAL CAIM  362.427424376964
4085.9098465442657


In [3]:
# Combinar los datos discretizados con la variable objetivo
# Es importante asegurarse de que los índices concuerden.
data_disc = x_disc.copy()
data_disc['class'] = y.values  # O, alternativamente: data_disc = pd.concat([x_disc, y.reset_index(drop=True)], axis=1)

# Guardar el DataFrame resultante en un CSV
ruta_guardado = "C:\\Users\\Carlo\\Desktop\\IA\\MDLP\\base de datos discretizadas con CAIM\\dry-bean_caim.csv"
data_disc.to_csv(ruta_guardado, index=False)
print("Datos discretizados guardados en:", ruta_guardado)

Datos discretizados guardados en: C:\Users\Carlo\Desktop\IA\MDLP\base de datos discretizadas con CAIM\dry-bean_caim.csv
